# Lab 1 — The Modern GenAI Stack
**Day 1 Morning | ~45 minutes | Colab CPU**

---

## What You Will Build
By the end of this lab you will have:
1. Unwrapped a HuggingFace `pipeline` to see tokens → logits → probabilities
2. Called a hosted LLM (GPT-4o-mini) with the OpenAI client
3. Streamed tokens live to your terminal
4. Swapped the **same client code** to a different provider by changing one line
5. Chained a prompt template with LangChain

> **The key idea:** The OpenAI Chat Completions format became the de-facto wire protocol.
> Change `base_url` — keep everything else. This composes the whole course.

In [ ]:
%%capture
!pip install transformers torch sentence-transformers openai langchain langchain-openai httpx
print('Done')

## 🔑 Configuration
Paste the instructor-provided OpenAI API key below. Everything in this lab flows from this cell.

In [ ]:
# ─── CONFIGURATION — edit only this cell ───────────────────────────────────
# ── API KEY SETUP ────────────────────────────────────────────────────────
# Colab: left sidebar → 🔑 Secrets → '+ Add new secret'
# Name: OPENAI_API_KEY  |  Value: your key  |  Enable notebook access ✓
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
OPENAI_BASE_URL = 'https://api.openai.com/v1'  # default; we'll swap this later
DEFAULT_MODEL   = 'gpt-4o-mini'                # fast, cheap, great for demos
QUALITY_MODEL   = 'gpt-4o'                     # for the quality comparison cell
# ─────────────────────────────────────────────────────────────────────────────

configured = OPENAI_API_KEY != 'sk-PASTE_KEY_HERE'
print('Config loaded — key set:', OPENAI_API_KEY[:8] + '...' if configured else '⚠️  KEY NOT SET')

---

## Part A — The HuggingFace Pipeline (15 min)

We start local and small: a 124 M-parameter GPT-2 model that runs on Colab CPU.
The point is not the quality — the point is to **open the black box**.

In [ ]:
# Cell A1 — The simplest possible inference: 3 lines
# INSTRUCTOR NOTE: pause here. 'This is the entire AI industry abstracted into 3 lines.'
from transformers import pipeline

generator = pipeline('text-generation', model='gpt2')
result = generator('The best way to deploy a language model is', max_new_tokens=40)
print(result[0]['generated_text'])

In [ ]:
# Cell A2 — Unwrap the pipeline: text → token IDs
# INSTRUCTOR NOTE: 'Let us open the black box. Three steps happen every time.'
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(model_id)
model     = AutoModelForCausalLM.from_pretrained(model_id)

text   = 'The best way to deploy a language model is'
inputs = tokenizer(text, return_tensors='pt')

print('STEP 1 — TEXT → TOKENS')
print(f'  Input text : {text!r}')
print(f'  Token IDs  : {inputs["input_ids"].tolist()[0]}')
print(f'  Decoded    : {[tokenizer.decode([t]) for t in inputs["input_ids"][0]]}')
print(f'  Count      : {inputs["input_ids"].shape[1]} tokens')

In [ ]:
# Cell A3 — token IDs → logits → next-token probabilities
with torch.no_grad():
    outputs = model(**inputs)

logits          = outputs.logits
next_token_logits = logits[0, -1, :]          # predictions for the *next* token
probs           = torch.softmax(next_token_logits, dim=-1)
top5            = torch.topk(probs, 5)

print('STEP 2 — MODEL PREDICTS NEXT TOKEN')
print(f'  Vocab size : {logits.shape[-1]:,}')
print()
print('  Top 5 candidates:')
for score, idx in zip(top5.values, top5.indices):
    print(f'    {tokenizer.decode([idx.item()])!r:<20} {score.item()*100:.1f}%')

In [ ]:
# Cell A4 — Memory math: why model size matters for deployment
# INSTRUCTOR NOTE: connect this to the Memory slide (S053).

def memory_table(model, label):
    params  = sum(p.numel() for p in model.parameters())
    fp32_mb = params * 4 / 1e6
    fp16_mb = params * 2 / 1e6
    int4_mb = params * 0.5 / 1e6
    print(f'\n{label}')
    print(f'  Parameters : {params:,}  ({params/1e6:.0f}M)')
    print(f'  FP32       : {fp32_mb:.0f} MB')
    print(f'  FP16       : {fp16_mb:.0f} MB')
    print(f'  INT4       : {int4_mb:.0f} MB')
    print(f'  Scale-up   → a 7B model is FP16 ~14 GB, INT4 ~3.5 GB')

memory_table(model, 'GPT-2 (124M params)')

---

## Part B — Cloud LLM via the OpenAI Client (15 min)

> **The key pattern** — learn this once and it works with OpenAI, Groq, vLLM, 
> HuggingFace Inference Providers, LiteLLM, and the custom server you will build in Lab 4.
>
> ```python
> client = OpenAI(api_key=..., base_url=...)  # ← the only thing that changes
> ```

In [ ]:
# Cell B1 — The canonical OpenAI client call
# INSTRUCTOR NOTE: 'One pattern. Infinite backends. This line unlocks the whole course.'
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL)

response = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[
        {'role': 'system', 'content': 'You are an LLM deployment expert. Be concise.'},
        {'role': 'user',   'content': 'What are the top 3 reasons LLM demos fail in production?'}
    ]
)
print(response.choices[0].message.content)
print(f'\nTokens used: {response.usage.total_tokens}')

In [ ]:
# Cell B2 — Streaming: watch tokens arrive live
# INSTRUCTOR NOTE: slow down here. Let everyone watch the tokens appear one by one.
print('Streaming response (tokens appear as generated):')
print('─' * 60)

stream = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[{'role': 'user', 'content': 'Explain the difference between FP16 and INT4 in 4 bullet points.'}],
    stream=True
)
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end='', flush=True)
print('\n' + '─' * 60)

In [ ]:
# Cell B3 — Quality comparison: same question, two models
# INSTRUCTOR NOTE: 'Change only the model name. The client does not care.'
question = 'In one sentence: what is the most important thing to know about LLM serving?'

r_mini = client.chat.completions.create(
    model=DEFAULT_MODEL, messages=[{'role': 'user', 'content': question}]
)
r_full = client.chat.completions.create(
    model=QUALITY_MODEL,  messages=[{'role': 'user', 'content': question}]
)

print(f'gpt-4o-mini : {r_mini.choices[0].message.content}')
print(f'gpt-4o      : {r_full.choices[0].message.content}')

In [ ]:
# Cell B4 — *** THE KEY MOMENT: swap base_url, keep everything else ***
# INSTRUCTOR NOTE: pause. 'Notice what changed: one string. Nothing else.'
# This also works with:
#   Groq:        base_url='https://api.groq.com/openai/v1', api_key='gsk_...'
#   vLLM:        base_url='http://your-gpu-server:8000/v1',  api_key='not-needed'
#   Lab 4 server:base_url='https://your-ngrok-url/v1',       api_key='not-needed'
#   LiteLLM GW:  base_url='http://gateway:4000/v1',          api_key='not-needed'

GROQ_BASE_URL   = 'https://api.groq.com/openai/v1'
GROQ_API_KEY    = 'gsk_PASTE_GROQ_KEY_IF_AVAILABLE'  # optional — demo only
GROQ_MODEL      = 'llama-3.1-8b-instant'

providers = {
    'OpenAI gpt-4o-mini': {
        'base_url': OPENAI_BASE_URL, 'api_key': OPENAI_API_KEY, 'model': DEFAULT_MODEL
    },
    # Uncomment below if instructor provides a Groq key:
    # 'Groq llama-3.1-8b': {
    #     'base_url': GROQ_BASE_URL, 'api_key': GROQ_API_KEY, 'model': GROQ_MODEL
    # },
}

q = 'In one sentence: what is PagedAttention?'
for name, cfg in providers.items():
    c = OpenAI(api_key=cfg['api_key'], base_url=cfg['base_url'])
    r = c.chat.completions.create(model=cfg['model'], messages=[{'role': 'user', 'content': q}])
    print(f'[{name}]\n  {r.choices[0].message.content}\n')

---

## Part C — LangChain (15 min)

LangChain also uses the OpenAI client interface.
Same `base_url` pattern — swap backends the same way.

In [ ]:
# Cell C1 — ChatOpenAI with our Groq/OpenAI backend
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_BASE_URL,
    model=DEFAULT_MODEL
)

messages = [
    SystemMessage(content='You are an LLM deployment expert. Be concise.'),
    HumanMessage(content='What is the difference between vLLM and a simple FastAPI server for LLM inference?')
]
print(llm.invoke(messages).content)

In [ ]:
# Cell C2 — Prompt templates + chaining: the LCEL pipe operator
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

template = ChatPromptTemplate.from_messages([
    ('system', 'You are a technical instructor. Explain concepts clearly with one analogy.'),
    ('user',   'Explain {concept} to someone who {background}.')
])

chain = template | llm | StrOutputParser()

print(chain.invoke({
    'concept':    'quantization',
    'background': 'has never worked with neural networks before'
}))

---

## ✅ Lab 1 Complete

You should now have:
- [ ] GPT-2 token IDs and top-5 next-token probabilities printed
- [ ] Memory table for GPT-2 (FP32 / FP16 / INT4)
- [ ] A streaming response from GPT-4o-mini
- [ ] Two-model quality comparison
- [ ] LangChain chain producing an analogy

## Stretch Goals

1. **Token counting:** Call the same prompt at three temperatures (0, 0.5, 1.0). 
   What changes? What stays the same?
2. **Browse the Hub:** Go to `huggingface.co/models?pipeline_tag=text-generation`. 
   Find a model fine-tuned for SQL generation. What is its parameter count?
3. **Logprobs:** Add `logprobs=True` to a non-streaming OpenAI call. 
   Print the top-5 token probabilities for the first word of the response.
4. **Add Groq:** If the instructor provides a Groq key, uncomment the Groq provider 
   in Cell B4. Compare speed vs GPT-4o-mini on the same prompt.
5. **Memory math:** Compute memory estimates for a 7B, 13B, and 70B model at FP16 and INT4. 
   Which fits in a T4 GPU (16 GB)?